# Issue #7: TensorFlow seed42 validation-only baseline on Kaggle T4

This notebook implements Protocol Amendment A for GitHub Issue #7. It
runs the frozen OFIX7-mid seed42 baseline through training and validation,
stops before final checkpoint loading or test-data construction, verifies
the validation-only completion marker, and immediately audits learning
history. The legacy `kaggle-end-to-end.ipynb` is intentionally not used as
an executable runner because it invokes the normal final-test lifecycle.

Required Kaggle Inputs:

- FER split dataset: `/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split`
  (only `train.csv` and `val.csv` are inspected).
- MediaPipe priors: `/kaggle/input/datasets/irthn1311/d16-mediapipe-pixel-priors-best-retry-rescue/outputs/d16_mediapipe_pixel_priors_best_retry_rescue`
  (only the `train` and `val` directories plus shared schema files are inspected).
- Clean graph cache: `/kaggle/input/datasets/irthn1311/ofix7-mid-seed42-records`
  (only `train/index.json`, `val/index.json`, and their referenced shards are inspected).

A Kaggle GPU T4 session and Internet access are required. Internet is used
only to clone the exact repository commit and, when necessary, install the
registered Python dependencies. All FER/prior/cache assets are offline
Kaggle Inputs. The final archive is
`/kaggle/working/tf_step4_seed42_validation_only_kaggle_t4.zip`.


## 1. Registered constants


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/FER2013_Graph.git"
REPO_BRANCH = "main"
EXPECTED_COMMIT = "4e3a80525a33679fd9ea8e19a85807d19736c981"
TF_PACKAGE_RELATIVE = Path("standalone/lap_gnn_tensorflow_ofix7_mid_candidate")

FER_SPLIT_ROOT = Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split")
FER_CSV_PATH = FER_SPLIT_ROOT / "train.csv"
PRIOR_ROOT = Path("/kaggle/input/datasets/irthn1311/d16-mediapipe-pixel-priors-best-retry-rescue/outputs/d16_mediapipe_pixel_priors_best_retry_rescue")
GRAPH_CACHE_ROOT = Path("/kaggle/input/datasets/irthn1311/ofix7-mid-seed42-records")

RUN_ID = "issue7_seed42_kaggle_t4"
WORKING = Path("/kaggle/working")
PROJECT_PATH = WORKING / "FER2013_Graph"
TF_PACKAGE_PATH = PROJECT_PATH / TF_PACKAGE_RELATIVE
OUTPUT_ROOT = WORKING / "outputs/tf_root_cause/step4_seed42_validation_only" / RUN_ID
AUDIT_ROOT = WORKING / "outputs/tf_root_cause/step4_seed42_audit" / RUN_ID
METADATA_ROOT = WORKING / "tf_step4_seed42_metadata"
ARCHIVE_PATH = Path("/kaggle/working/tf_step4_seed42_validation_only_kaggle_t4.zip")
EVIDENCE_JSON = WORKING / "tf_step4_seed42_validation_only_kaggle_t4_evidence.json"
EVIDENCE_REPORT = WORKING / "tf_step4_seed42_validation_only_baseline.md"

TRAIN_CONFIG_RELATIVE = Path("configs/fer2013_ofix7_mid_tensorflow_seed42.yaml")
EXPECTED_CONFIG_SHA256 = "aa3bf2d3932bbad6c5f8cdcc347f4a9866e2c027d6135a60b5002a8f6a3b6908"
EXPECTED_WRAPPER_SHA256 = "c94c122066fdd19210c8ba64a2a61567b249fad4f69c69cb4236b68cce6ff7b4"
EXPECTED_TRAINER_SHA256 = "4c3cb1aa311578038ff656cb7d119103ae5a651135f8ee1c76e37c2c04c1fc75"
EXPECTED_SCIENTIFIC_PAYLOAD_SHA256 = "286be711a53b76511bcf3b9bf949fad694f7c7d272392f9defc56f4914822c0e"
EXPECTED_EXECUTION_CONTRACT_SHA256 = "14acc2750875a25922007459161a137158d8040805e616166be923f63658bf22"

SEED = 42
DEVICE = "gpu"
GRAPH_WORKERS = 2
TF_DATA_PREFETCH = 2
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
MIXED_PRECISION = True
XLA_ENABLED = False
MEMORY_GROWTH = True
RESUME = False

EXCLUDED_LOCAL_EXECUTION = {
    "status": "excluded_by_protocol_amendment_a",
    "attempt_1": "local GPU OOM before any completed epoch",
    "bounded_smoke": "local technical allocator smoke; not scientific evidence",
    "attempt_2": "stopped when Amendment A locked the baseline to Kaggle T4",
}

assert SEED == 42 and DEVICE == "gpu"
assert GRAPH_WORKERS == 2 and TF_DATA_PREFETCH == 2
assert BATCH_SIZE == 16 and EVAL_BATCH_SIZE == 32
assert MIXED_PRECISION and MEMORY_GROWTH
assert not XLA_ENABLED and not RESUME


## 2. Clone the exact preregistered source and verify frozen hashes


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys

def run_checked(command, cwd=None, env=None, capture=False):
    actual = [str(item) for item in command]
    display = [re.sub(r"(https://x-access-token:)[^@]+@", r"\1***@", item) for item in actual]
    print("$", " ".join(display))
    result = subprocess.run(
        actual,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    if result.returncode:
        if capture and result.stdout:
            print("\n".join(result.stdout.splitlines()[-100:]))
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout if capture else ""

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

if PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://x-access-token:{github_token}@")
run_checked(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", clone_url, PROJECT_PATH])
run_checked(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=PROJECT_PATH)
actual_commit = run_checked(["git", "rev-parse", "HEAD"], cwd=PROJECT_PATH, capture=True).strip()
dirty = run_checked(["git", "status", "--porcelain"], cwd=PROJECT_PATH, capture=True).strip()
if actual_commit != EXPECTED_COMMIT or dirty:
    raise RuntimeError(f"Source lock failed: commit={actual_commit}, dirty={bool(dirty)}")

TF_PACKAGE_PATH = PROJECT_PATH / TF_PACKAGE_RELATIVE
TRAIN_CONFIG = TF_PACKAGE_PATH / TRAIN_CONFIG_RELATIVE
WRAPPER_PATH = TF_PACKAGE_PATH / "tools/train_validation_only.py"
AUDIT_TOOL_PATH = TF_PACKAGE_PATH / "tools/audit_learning_history.py"
TRAINER_PATH = TF_PACKAGE_PATH / "src/lap_gnn_tf/training/trainer.py"
for required in [
    TF_PACKAGE_PATH / "pyproject.toml",
    TF_PACKAGE_PATH / "CHECKSUMS.sha256",
    TF_PACKAGE_PATH / "package_manifest.json",
    TRAIN_CONFIG,
    WRAPPER_PATH,
    AUDIT_TOOL_PATH,
    TRAINER_PATH,
]:
    if not required.is_file():
        raise FileNotFoundError(required)

run_checked([sys.executable, "-B", TF_PACKAGE_PATH / "tools/verify_checksums.py"], cwd=TF_PACKAGE_PATH)
manifest = json.loads((TF_PACKAGE_PATH / "package_manifest.json").read_text(encoding="utf-8"))
if manifest.get("scientific_payload_sha256") != EXPECTED_SCIENTIFIC_PAYLOAD_SHA256:
    raise RuntimeError("Frozen scientific payload drift")
if manifest.get("execution_contract_sha256") != EXPECTED_EXECUTION_CONTRACT_SHA256:
    raise RuntimeError("Execution contract drift")
if sha256(TRAIN_CONFIG) != EXPECTED_CONFIG_SHA256:
    raise RuntimeError("Preregistered seed42 config drift")
if sha256(WRAPPER_PATH) != EXPECTED_WRAPPER_SHA256:
    raise RuntimeError("Validation-only wrapper drift")
if sha256(TRAINER_PATH) != EXPECTED_TRAINER_SHA256:
    raise RuntimeError("Frozen trainer drift")
print(json.dumps({
    "commit": actual_commit,
    "config_sha256": sha256(TRAIN_CONFIG),
    "wrapper_sha256": sha256(WRAPPER_PATH),
    "trainer_sha256": sha256(TRAINER_PATH),
    "scientific_payload_sha256": manifest["scientific_payload_sha256"],
}, indent=2))


## 3. Install the registered TensorFlow environment and require Kaggle T4


In [ ]:
import importlib.metadata
import importlib.util
import platform

def distribution_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

print("python:", sys.version)
print("platform:", platform.platform(), platform.machine())
print("disk_free_gib:", round(shutil.disk_usage(WORKING).free / 2**30, 2))
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError("TensorFlow/Keras imported before environment bootstrap")
gpu_names = [
    line.strip() for line in run_checked(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture=True
    ).splitlines() if line.strip()
]
if not gpu_names or not all("T4" in name for name in gpu_names):
    raise RuntimeError(f"Protocol Amendment A requires Kaggle GPU T4, got {gpu_names}")
run_checked(["nvidia-smi"])

TESTED_TF = "2.18.1"
TESTED_KERAS = "3.15.0"
if distribution_version("tensorflow") != TESTED_TF or distribution_version("keras") != TESTED_KERAS:
    run_checked([
        sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "-r",
        TF_PACKAGE_PATH / "requirements-kaggle.txt",
    ])
    importlib.invalidate_caches()
missing = [
    requirement for module, requirement in [
        ("yaml", "PyYAML==6.0.2"),
        ("sklearn", "scikit-learn==1.6.1"),
        ("psutil", "psutil==6.1.1"),
        ("matplotlib", "matplotlib==3.10.0"),
    ] if importlib.util.find_spec(module) is None
]
if missing:
    run_checked([sys.executable, "-m", "pip", "install", "-q", *missing])
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", TF_PACKAGE_PATH, "--no-deps"])

probe_text = run_checked([
    sys.executable, "-B", "-c",
    "import json,tensorflow as tf; print(json.dumps({"
    "'tensorflow':tf.__version__,'keras':tf.keras.__version__,"
    "'gpus':[d.name for d in tf.config.list_physical_devices('GPU')],"
    "'build':tf.sysconfig.get_build_info()},default=str))",
], cwd=TF_PACKAGE_PATH, capture=True)
probe_payload = json.loads(probe_text.strip().splitlines()[-1])
if probe_payload["tensorflow"] != TESTED_TF or probe_payload["keras"] != TESTED_KERAS:
    raise RuntimeError(f"Registered TensorFlow/Keras mismatch: {probe_payload}")
if not probe_payload["gpus"]:
    raise RuntimeError("TensorFlow does not expose the required T4")
METADATA_ROOT.mkdir(parents=True, exist_ok=False)
environment_payload = {
    "python": sys.version,
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "tensorflow": probe_payload["tensorflow"],
    "keras": probe_payload["keras"],
    "tensorflow_build": probe_payload["build"],
    "gpu_devices": probe_payload["gpus"],
    "nvidia_gpu_names": gpu_names,
    "nvidia_smi_query": run_checked([
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ], capture=True).splitlines(),
}
(METADATA_ROOT / "tensorflow_environment.json").write_text(
    json.dumps(environment_payload, indent=2, default=str) + "\n",
    encoding="utf-8",
)
(METADATA_ROOT / "kaggle_gpu.json").write_text(
    json.dumps({"gpu_names": gpu_names, "probe": probe_payload}, indent=2, default=str),
    encoding="utf-8",
)
print(json.dumps(environment_payload, indent=2, default=str))


## 4. Import isolation and bounded implementation preflight


In [ ]:
PACKAGE_SRC = TF_PACKAGE_PATH / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))
importlib.invalidate_caches()

import tensorflow as tf
import lap_gnn_tf
from lap_gnn_tf.training.execution import configure_restricted_grappler

resolved_package = Path(lap_gnn_tf.__file__).resolve()
if TF_PACKAGE_PATH.resolve() not in resolved_package.parents:
    raise RuntimeError(f"lap_gnn_tf resolved outside frozen package: {resolved_package}")
if "torch" in sys.modules:
    raise RuntimeError("TensorFlow notebook imported torch")
run_checked([sys.executable, "-B", TF_PACKAGE_PATH / "tools/verify_no_torch_runtime.py"], cwd=TF_PACKAGE_PATH)
run_checked([sys.executable, "-B", TF_PACKAGE_PATH / "tools/verify_no_parent_imports.py"], cwd=TF_PACKAGE_PATH)
run_checked([
    sys.executable, "-B", "-m", "pytest", "-q",
    "tests/test_train_validation_only.py",
    "tests/test_audit_learning_history.py",
    "tests/test_scientific_payload_checksum_portable.py",
    "tests/test_no_parent_imports.py",
    "tests/test_no_torch_runtime.py",
], cwd=TF_PACKAGE_PATH)

effective_grappler = configure_restricted_grappler()
for key in ("arithmetic_optimization", "remapping"):
    if effective_grappler.get(key) is not False:
        raise RuntimeError(f"Registered G1-A option drift: {key}")
parity_env = os.environ.copy()
parity_env["CUDA_VISIBLE_DEVICES"] = ""
parity_env["TF_DETERMINISTIC_OPS"] = "1"
PARITY_REPORT = METADATA_ROOT / "tensorflow_golden_parity.json"
run_checked([
    sys.executable, "-B", "-m", "lap_gnn_tf.cli.compare_golden",
    "--package-root", TF_PACKAGE_PATH,
    "--output", PARITY_REPORT,
], cwd=TF_PACKAGE_PATH, env=parity_env)
parity = json.loads(PARITY_REPORT.read_text(encoding="utf-8"))
if not parity.get("pass"):
    raise RuntimeError(f"Golden parity failed: {parity}")
print("READY_FOR_ISSUE7_KAGGLE_T4_ASSET_PREFLIGHT")


## 5. Train/validation-only asset and policy preflight


In [ ]:
import csv
from lap_gnn_tf.config import load_config, validate_locked_config
from lap_gnn_tf.constants import EDGE_FEATURE_NAMES, NODE_FEATURE_NAMES, EXPECTED_PARAMETER_COUNT
from lap_gnn_tf.signatures import scientific_payload_checksum

allowed_splits = {"train": 28709, "val": 3589}
asset_evidence = {}
for split, expected_count in allowed_splits.items():
    csv_path = FER_SPLIT_ROOT / f"{split}.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(csv_path)
    with csv_path.open("r", encoding="utf-8", newline="") as stream:
        reader = csv.reader(stream)
        header = [item.strip().lower() for item in next(reader)]
        row_count = sum(1 for _ in reader)
    if "emotion" not in header or "pixels" not in header or row_count != expected_count:
        raise RuntimeError(f"{split} CSV mismatch: rows={row_count}, header={header}")

    prior_split = PRIOR_ROOT / split
    prior_count = len(list(prior_split.glob("*.npz")))
    if prior_count != expected_count:
        raise RuntimeError(f"{split} prior count mismatch: {prior_count} != {expected_count}")

    cache_index_path = GRAPH_CACHE_ROOT / split / "index.json"
    if not cache_index_path.is_file():
        raise FileNotFoundError(cache_index_path)
    cache_index = json.loads(cache_index_path.read_text(encoding="utf-8"))
    if cache_index.get("schema_version") != "tf_clean_graph_cache_v2_records":
        raise RuntimeError(f"{split} cache schema mismatch")
    if int(cache_index.get("sample_count", -1)) != expected_count:
        raise RuntimeError(f"{split} cache count mismatch")
    for shard in cache_index.get("shards", []):
        shard_path = GRAPH_CACHE_ROOT / split / shard["path"]
        if not shard_path.is_file():
            raise FileNotFoundError(shard_path)
    asset_evidence[split] = {
        "csv_path": str(csv_path),
        "csv_sha256": sha256(csv_path),
        "csv_rows": row_count,
        "prior_path": str(prior_split),
        "prior_files": prior_count,
        "cache_index_path": str(cache_index_path),
        "cache_index_sha256": sha256(cache_index_path),
        "cache_samples": int(cache_index["sample_count"]),
        "cache_shards": len(cache_index.get("shards", [])),
    }

schema_path = PRIOR_ROOT / "prior_schema.json"
schema_payload = json.loads(schema_path.read_text(encoding="utf-8"))
if schema_payload.get("schema_version") != "d16_mediapipe_pixel_priors_v1":
    raise RuntimeError(
        f"Prior schema mismatch: {schema_payload.get('schema_version')}"
    )
if len(NODE_FEATURE_NAMES) != 37 or len(EDGE_FEATURE_NAMES) != 8:
    raise RuntimeError("Frozen feature dimensions drift")
if EXPECTED_PARAMETER_COUNT != 1_061_192:
    raise RuntimeError("Frozen parameter count drift")

locked_config = load_config(TRAIN_CONFIG)
validate_locked_config(locked_config)
training = locked_config["training"]
if locked_config["seed"] != 42:
    raise RuntimeError("Seed drift")
if training["max_epochs"] != 90:
    raise RuntimeError("Max-epoch drift")
if training["checkpoint_monitor"] != "val_accuracy" or training["best_metric"] != "val_accuracy":
    raise RuntimeError("Checkpoint/model-selection drift")
if training["scheduler"]["monitor"] != "val_loss":
    raise RuntimeError("Scheduler monitor drift")
if training["early_stopping"]["metric"] != "val_loss":
    raise RuntimeError("Early-stop monitor drift")
if training["early_stopping"]["min_epochs_before_stop"] != 30 or training["early_stopping"]["patience"] != 15:
    raise RuntimeError("Early-stop policy drift")
if training.get("eval_train_every_n_epochs", 1) != 1:
    raise RuntimeError("Issue #7 requires clean train evaluation every epoch")
if locked_config["locked"]["package_checksum"] != EXPECTED_SCIENTIFIC_PAYLOAD_SHA256:
    raise RuntimeError("Config payload lock drift")
if scientific_payload_checksum(TF_PACKAGE_PATH) != EXPECTED_SCIENTIFIC_PAYLOAD_SHA256:
    raise RuntimeError("Live scientific payload drift")
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Kaggle T4 disappeared before the registered run")

for fresh_path in (OUTPUT_ROOT, AUDIT_ROOT, ARCHIVE_PATH, EVIDENCE_JSON, EVIDENCE_REPORT):
    if fresh_path.exists():
        raise FileExistsError(f"Registered output must be fresh: {fresh_path}")
(METADATA_ROOT / "allowed_asset_evidence.json").write_text(
    json.dumps({
        "splits_inspected": list(allowed_splits),
        "assets": asset_evidence,
        "prior_schema_path": str(schema_path),
        "prior_schema_sha256": sha256(schema_path),
        "test_assets_inspected": False,
    }, indent=2),
    encoding="utf-8",
)
print(json.dumps(asset_evidence, indent=2))
print("READY_FOR_ISSUE7_REGISTERED_BASELINE")


## 6. One registered full validation-only baseline


In [ ]:
train_command = [
    sys.executable, "-B", WRAPPER_PATH,
    "--config", TRAIN_CONFIG,
    "--fer-csv", FER_CSV_PATH,
    "--prior-root", PRIOR_ROOT,
    "--output-root", OUTPUT_ROOT,
    "--device", DEVICE,
    "--graph-workers", str(GRAPH_WORKERS),
    "--tf-data-prefetch", str(TF_DATA_PREFETCH),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--mixed-precision",
    "--no-xla",
    "--memory-growth",
    "--no-resume",
    "--clean-graph-cache-dir", GRAPH_CACHE_ROOT,
]
if any(str(argument).startswith("--limit-") for argument in train_command):
    raise RuntimeError("Registered full baseline must not use bounded limits")
run_checked(train_command, cwd=TF_PACKAGE_PATH)


## 7. Fail-closed validation-only boundary verification


In [ ]:
MARKER_PATH = OUTPUT_ROOT / "VALIDATION_ONLY_COMPLETE.json"
if not MARKER_PATH.is_file():
    raise RuntimeError("Validation-only completion marker is missing")
marker = json.loads(MARKER_PATH.read_text(encoding="utf-8"))
required_true = ["training_validation_completed", "final_test_skipped", "trainer_revision_guard_passed"]
required_false = [
    "test_accessed", "test_data_constructed", "test_checkpoint_loaded",
    "normal_full_training_completed",
]
if any(marker.get(key) is not True for key in required_true):
    raise RuntimeError(f"Validation-only true-field failure: {marker}")
if any(marker.get(key) is not False for key in required_false):
    raise RuntimeError(f"Validation-only false-field failure: {marker}")
if marker.get("scientific_payload_sha256") != EXPECTED_SCIENTIFIC_PAYLOAD_SHA256:
    raise RuntimeError("Marker scientific payload drift")
if marker.get("trainer_source_sha256") != EXPECTED_TRAINER_SHA256:
    raise RuntimeError("Marker trainer revision drift")
if any(value is not None for value in marker.get("bounded_limits", {}).values()):
    raise RuntimeError("Registered full baseline unexpectedly used a limit")

required_run_artifacts = [
    OUTPUT_ROOT / "history.json",
    OUTPUT_ROOT / "resolved_config.json",
    OUTPUT_ROOT / "resolved_config.yaml",
    OUTPUT_ROOT / "telemetry.json",
    OUTPUT_ROOT / "provenance.json",
    MARKER_PATH,
]
missing = [str(path) for path in required_run_artifacts if not path.is_file()]
if missing:
    raise RuntimeError(f"Required validation-only artifacts missing: {missing}")
forbidden_exact = [
    OUTPUT_ROOT / "TRAINING_COMPLETE.json",
    OUTPUT_ROOT / "run_summary.json",
    OUTPUT_ROOT / "predictions.csv",
    OUTPUT_ROOT / "per_class_metrics.csv",
    OUTPUT_ROOT / "confusion_matrix.csv",
    OUTPUT_ROOT / "confusion_matrix.png",
]
forbidden_found = [str(path) for path in forbidden_exact if path.exists()]
forbidden_found.extend(str(path) for path in OUTPUT_ROOT.glob("test_metrics_*.json"))
if forbidden_found:
    raise RuntimeError(f"Forbidden post-test artifacts exist: {forbidden_found}")
print(json.dumps(marker, indent=2, sort_keys=True))
print("VALIDATION_ONLY_BOUNDARY_VERIFIED")


## 8. Immediate read-only learning-history audit and compact report


In [ ]:
audit_command = [
    sys.executable, "-B", AUDIT_TOOL_PATH,
    "--run-dir", OUTPUT_ROOT,
    "--output-dir", AUDIT_ROOT,
]
run_checked(audit_command, cwd=TF_PACKAGE_PATH)
audit_files = [
    AUDIT_ROOT / "learning_diagnosis.json",
    AUDIT_ROOT / "learning_diagnosis.md",
    AUDIT_ROOT / "epoch_metrics_validation_only.csv",
]
missing_audit = [str(path) for path in audit_files if not path.is_file()]
if missing_audit:
    raise RuntimeError(f"Learning-history audit artifacts missing: {missing_audit}")

diagnosis = json.loads((AUDIT_ROOT / "learning_diagnosis.json").read_text(encoding="utf-8"))
if diagnosis["provenance"].get("test_artifacts_read") is not False:
    raise RuntimeError("Audit provenance does not prove test isolation")
history_rows = json.loads((OUTPUT_ROOT / "history.json").read_text(encoding="utf-8"))["epochs"]
resolved_config = json.loads((OUTPUT_ROOT / "resolved_config.json").read_text(encoding="utf-8"))
measurements = diagnosis["measurements"]
policy = diagnosis["policy"]
interpretation = diagnosis["interpretation"]

lr_trajectory = [{"epoch": int(row["epoch"]), "lr": float(row["lr"])} for row in history_rows]
lr_reductions = [
    current for previous, current in zip(lr_trajectory, lr_trajectory[1:])
    if current["lr"] < previous["lr"]
]
prior_schedule = resolved_config["graph"].get("prior_corruption", {})
total_epochs = len(history_rows)
max_epochs = int(resolved_config["training"]["max_epochs"])
stop_reason = "early_stopping" if bool(history_rows[-1].get("stop_requested")) else "max_epochs"
if stop_reason == "max_epochs" and total_epochs != max_epochs:
    stop_reason = "UNKNOWN_INCOMPLETE"

runtime_hashes = {
    "history_sha256": sha256(OUTPUT_ROOT / "history.json"),
    "resolved_config_sha256": sha256(OUTPUT_ROOT / "resolved_config.json"),
    "marker_sha256": sha256(MARKER_PATH),
    "telemetry_sha256": sha256(OUTPUT_ROOT / "telemetry.json"),
}
hypothesis_map = {
    "GENERALIZATION_GAP_SIGNAL": "H-GEN",
    "MODERATE_GENERALIZATION_GAP": "H-MIXED",
    "SMALL_GENERALIZATION_GAP": "H-REP",
    "UNKNOWN_TRAIN_EVAL_INCOMPLETE": "UNKNOWN",
}
environment_payload = json.loads(
    (METADATA_ROOT / "tensorflow_environment.json").read_text(encoding="utf-8")
)
evidence = {
    "issue": 7,
    "protocol_amendment": "A",
    "registered_environment": "Kaggle GPU T4",
    "base_commit": EXPECTED_COMMIT,
    "config_path": str(TRAIN_CONFIG_RELATIVE),
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "wrapper_version": marker["wrapper_version"],
    "wrapper_sha256": EXPECTED_WRAPPER_SHA256,
    "trainer_sha256": EXPECTED_TRAINER_SHA256,
    "scientific_payload_sha256": EXPECTED_SCIENTIFIC_PAYLOAD_SHA256,
    "seed": SEED,
    "exact_command": [str(item) for item in train_command],
    "kaggle_assets": {
        "fer_split_dataset": "doduyquynii/fer13-split",
        "mediapipe_prior_dataset": "irthn1311/d16-mediapipe-pixel-priors-best-retry-rescue",
        "clean_graph_cache_dataset": "irthn1311/ofix7-mid-seed42-records",
        "splits_inspected": ["train", "val"],
        "test_assets_inspected": False,
    },
    "excluded_local_execution": EXCLUDED_LOCAL_EXECUTION,
    "kaggle_bounded_smoke_used": False,
    "runtime_hashes": runtime_hashes,
    "runtime_integrity": {
        "total_epochs": total_epochs,
        "stop_reason": stop_reason,
        "marker": marker,
        "environment": environment_payload,
        "test_phase_ran": False,
    },
    "measurements": measurements,
    "lr_trajectory": lr_trajectory,
    "lr_reductions": lr_reductions,
    "prior_corruption_schedule": prior_schedule,
    "policy": policy,
    "interpretation": interpretation,
    "preregistered_hypothesis_label": hypothesis_map.get(
        interpretation["learning_behavior"], "UNKNOWN"
    ),
}
EVIDENCE_JSON.write_text(
    json.dumps(evidence, indent=2, sort_keys=True, ensure_ascii=True) + "\n",
    encoding="utf-8",
)

def display(value):
    if value is None:
        return "UNKNOWN"
    if isinstance(value, float):
        return f"{value:.6f}"
    return str(value)

report = f'''# TensorFlow Step 4 seed42 validation-only baseline

## Preregistration provenance

- Issue: `#7`, including Protocol Amendment A.
- Registered environment: Kaggle GPU T4.
- Base commit: `{EXPECTED_COMMIT}`.
- Config: `{TRAIN_CONFIG_RELATIVE}`; SHA-256 `{EXPECTED_CONFIG_SHA256}`.
- Wrapper: version `{marker['wrapper_version']}`; SHA-256 `{EXPECTED_WRAPPER_SHA256}`.
- Trainer SHA-256: `{EXPECTED_TRAINER_SHA256}`.
- Scientific payload SHA-256: `{EXPECTED_SCIENTIFIC_PAYLOAD_SHA256}`.
- Seed: `{SEED}`.
- Exact command: `{' '.join(str(item) for item in train_command)}`.
- Allowed inputs: FER train/validation CSVs, MediaPipe train/validation priors, and train/validation clean-cache shards from the three registered Kaggle datasets.
- Kaggle bounded smoke used: `false`.
- Pre-amendment local attempts: excluded from scientific evidence; the surviving local process was stopped when Amendment A was received.
- History SHA-256: `{runtime_hashes['history_sha256']}`.
- Resolved-config SHA-256: `{runtime_hashes['resolved_config_sha256']}`.
- Validation-only marker SHA-256: `{runtime_hashes['marker_sha256']}`.

## Runtime integrity

- Total epochs: `{total_epochs}`.
- Stopping reason: `{stop_reason}`.
- `training_validation_completed`: `{str(marker['training_validation_completed']).lower()}`.
- `final_test_skipped`: `{str(marker['final_test_skipped']).lower()}`.
- `test_accessed`: `{str(marker['test_accessed']).lower()}`.
- `test_data_constructed`: `{str(marker['test_data_constructed']).lower()}`.
- `test_checkpoint_loaded`: `{str(marker['test_checkpoint_loaded']).lower()}`.
- Test phase ran: `false`.
- Environment: TensorFlow `{environment_payload.get('tensorflow')}` on `{environment_payload.get('gpu_devices')}`.

## Raw measurements

- Best validation loss: epoch `{measurements['best_validation_loss']['epoch']}`, value `{display(measurements['best_validation_loss']['value'])}`.
- Best validation accuracy: epoch `{measurements['best_validation_accuracy']['epoch']}`, value `{display(measurements['best_validation_accuracy']['value'])}`.
- Best validation macro-F1: epoch `{measurements['best_validation_macro_f1']['epoch']}`, value `{display(measurements['best_validation_macro_f1']['value'])}`.
- Best-epoch spread: `{measurements['best_epoch_spread']}`.
- Train macro-F1 at best validation macro-F1: `{display(measurements['train_macro_f1_at_best_validation_macro_f1'])}`.
- Primary train-validation macro-F1 gap: `{display(measurements['train_validation_macro_f1_gap_pp_at_best_validation_macro_f1'])}` pp.
- Final train macro-F1: `{display(measurements['final_train_macro_f1'])}`.
- Final validation macro-F1: `{display(measurements['final_validation_macro_f1'])}`.
- Final gap: `{display(measurements['final_train_validation_macro_f1_gap_pp'])}` pp.
- Learning-rate reductions: `{json.dumps(lr_reductions, sort_keys=True)}`.
- Prior-corruption schedule: `{json.dumps(prior_schedule, sort_keys=True)}`.

## Policy evidence

- Checkpoint monitor: `{display(policy['checkpoint_monitor'])}`.
- Scheduler monitor: `{display(policy['scheduler_monitor'])}`.
- Early-stop monitor: `{display(policy['early_stopping_monitor'])}`.
- Final-test checkpoint config field: `{display(policy['final_test_checkpoint'])}` (not loaded or evaluated).
- Model-selection metric: `{display(policy['final_model_selection_metric'])}`.
- Monitor-policy label: `{interpretation['monitor_policy']}`.

## Diagnostic outcome

- Learning label: `{interpretation['learning_behavior']}`.
- Preregistered hypothesis mapping: `{hypothesis_map.get(interpretation['learning_behavior'], 'UNKNOWN')}`.

## Scientific interpretation boundary

This result is a validation-only diagnostic. The preregistered gap label and
monitor-policy label are evidence summaries, not proof of overfitting or a
causal mechanism. The run does not evaluate or justify a change to the model,
graph, priors, features, loss, optimizer, scheduler, early stopping, checkpoint
policy, seed, or data split. No test data or test artifact influenced this report.
'''
EVIDENCE_REPORT.write_text(
    "\n".join(line[8:] if line.startswith("        ") else line for line in report.splitlines()) + "\n",
    encoding="utf-8",
)
print(json.dumps(evidence, indent=2, sort_keys=True, default=str))
print("evidence_report:", EVIDENCE_REPORT)


## 9. Archive validation-only evidence


In [ ]:
import zipfile

archive_sources = [OUTPUT_ROOT, AUDIT_ROOT, METADATA_ROOT, EVIDENCE_JSON, EVIDENCE_REPORT]
if ARCHIVE_PATH.exists():
    raise FileExistsError(f"Archive path is not fresh: {ARCHIVE_PATH}")
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for source in archive_sources:
        if source.is_dir():
            for path in sorted(source.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(WORKING))
        else:
            archive.write(source, source.relative_to(WORKING))
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archived_names = archive.namelist()
forbidden_archive_names = [
    name for name in archived_names
    if Path(name).name == "TRAINING_COMPLETE.json"
    or Path(name).name in {
        "predictions.csv", "per_class_metrics.csv",
        "confusion_matrix.csv", "confusion_matrix.png",
    }
    or (Path(name).name.startswith("test_metrics_") and Path(name).suffix == ".json")
]
if forbidden_archive_names:
    raise RuntimeError(f"Archive contains forbidden post-test artifacts: {forbidden_archive_names}")
print("archive:", ARCHIVE_PATH)
print("archive_bytes:", ARCHIVE_PATH.stat().st_size)
print("archive_sha256:", sha256(ARCHIVE_PATH))
print("archive_files:", len(archived_names))
